## Part 1 — Extract stock data into a DataFrame

In [38]:
import yfinance as yf
import pandas as pd

TICKERS = ["AAPL", "MSFT", "SPY", "^GSPC"]

stock_data = []

for ticker in TICKERS:
    history = yf.Ticker(ticker).history(period="1mo")
    history = history.reset_index()
    history["ticker"] = ticker
    stock_data.append(history)

stocks_df = pd.concat(stock_data, ignore_index=True)

display(stocks_df.head())

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,ticker,Capital Gains
0,2026-09-17 00:00:00-04:00,334.769989,338.339996,330.179993,337.000000,36700200,0.0,0.0,AAPL,NaN
1,2026-09-18 00:00:00-04:00,337.910004,338.489990,332.529999,336.130005,86588200,0.0,0.0,AAPL,NaN
2,2026-09-21 00:00:00-04:00,335.279999,339.640015,333.049988,338.980011,34999200,0.0,0.0,AAPL,NaN
3,2026-09-22 00:00:00-04:00,340.140015,345.339996,338.750000,339.750000,40711800,0.0,0.0,AAPL,NaN
4,2026-09-23 00:00:00-04:00,341.079987,341.799988,335.500000,337.019989,31658800,0.0,0.0,AAPL,NaN


## Part 2 — Load .env variables

In [39]:
from dotenv import load_dotenv
import os
import psycopg2

load_dotenv()

FRED_API_KEY = os.getenv("FRED_API_KEY")
SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")

## Part 3 — Extract FRED data into a DataFrame

In [40]:
import requests
import datetime as dt

url = "https://api.stlouisfed.org/fred/series/observations"

FRED_SERIES = ["FEDFUNDS", "CPIAUCSL", "UNRATE"]

START_DATE = (
    dt.date.today() - dt.timedelta(days=45)
).isoformat()

macro_data = []

for series_id in FRED_SERIES:
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": START_DATE
    }

    response = requests.get(url, params=params)
    data = response.json()

    for observation in data["observations"]:
        macro_data.append({
            "series_id": series_id,
            "observation_date": observation["date"],
            "value": observation["value"]
        })

macro_df = pd.DataFrame(macro_data)

display(macro_df.head())

,series_id,observation_date,value
0,FEDFUNDS,2026-09-01,3.75
1,CPIAUCSL,2026-09-01,336.298
2,UNRATE,2026-09-01,4.2


## Part 4 — Create the database tables

In [41]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
        CREATE TABLE IF NOT EXISTS raw_stock_prices (
            ticker TEXT NOT NULL,
            price_date DATE NOT NULL,
            open DOUBLE PRECISION,
            high DOUBLE PRECISION,
            low DOUBLE PRECISION,
            close DOUBLE PRECISION,
            volume BIGINT,
            source TEXT,
            ingested_at TIMESTAMPTZ DEFAULT NOW(),
            PRIMARY KEY (ticker, price_date)
        );

        CREATE TABLE IF NOT EXISTS raw_macro_indicators (
            series_id TEXT NOT NULL,
            observation_date DATE NOT NULL,
            value DOUBLE PRECISION,
            source TEXT,
            ingested_at TIMESTAMPTZ DEFAULT NOW(),
            PRIMARY KEY (series_id, observation_date)
        );
        """)

print("Tables created successfully!")

Tables created successfully!


## Part 5 — Insert stock data

In [42]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO raw_stock_prices
            (ticker, price_date, open, high, low, close, volume, source)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (ticker, price_date) DO NOTHING
        """, [
            (
                row["ticker"],
                row["Date"].date(),
                row["Open"],
                row["High"],
                row["Low"],
                row["Close"],
                int(row["Volume"]),
                "yfinance"
            )
            for _, row in stocks_df.iterrows()
        ])

print("Stock data inserted successfully!")

Stock data inserted successfully!


## Part 6 — Insert FRED data

In [43]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO raw_macro_indicators
            (series_id, observation_date, value, source)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (series_id, observation_date) DO NOTHING
        """, [
            (
                row["series_id"],
                row["observation_date"],
                float(row["value"]) if row["value"] != "." else None,
                "FRED"
            )
            for _, row in macro_df.iterrows()
        ])

print("FRED data inserted successfully!")

FRED data inserted successfully!


## Part 7.1 — Create staging views

In [44]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE OR REPLACE VIEW stg_stock_prices AS
            SELECT
                ticker,
                price_date,
                open,
                high,
                low,
                close,
                volume,
                source,
                ingested_at
            FROM raw_stock_prices;

            CREATE OR REPLACE VIEW stg_macro_indicators AS
            SELECT
                series_id,
                observation_date,
                value,
                source,
                ingested_at
            FROM raw_macro_indicators;
        """)

print("Staging views created successfully!")

Staging views created successfully!


## Part 7.2 — Create dimension tables

## dim_ticker

In [45]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS dim_ticker AS
            SELECT DISTINCT ticker
            FROM stg_stock_prices;
        """)

print("dim_ticker created successfully!")

dim_ticker created successfully!


## dim_date

In [46]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS dim_date AS
            SELECT date_key
            FROM (
                SELECT price_date AS date_key
                FROM stg_stock_prices

                UNION

                SELECT observation_date AS date_key
                FROM stg_macro_indicators
            ) AS dates
            WHERE date_key IS NOT NULL;
        """)

print("dim_date created successfully!")

dim_date created successfully!


## Part 7.3 — Create fact tables

## fact_stock_price

In [47]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS fact_stock_price AS
            SELECT
                ticker,
                price_date AS date_key,
                open,
                high,
                low,
                close,
                volume
            FROM stg_stock_prices;
        """)

print("fact_stock_price created successfully!")

fact_stock_price created successfully!


## fact_macro_indicator

In [48]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS fact_macro_indicator AS
            SELECT
                series_id,
                observation_date AS date_key,
                value
            FROM stg_macro_indicators;
        """)

print("fact_macro_indicator created successfully!")

fact_macro_indicator created successfully!


## Part 7.4 — Check the tables

In [49]:
with psycopg2.connect(SUPABASE_DB_URL) as conn:
    with conn.cursor() as cur:
        for table in [
            "stg_stock_prices",
            "stg_macro_indicators",
            "dim_ticker",
            "dim_date",
            "fact_stock_price",
            "fact_macro_indicator"
        ]:
            cur.execute(f"SELECT COUNT(*) FROM {table}")
            print(f"{table}: {cur.fetchone()[0]} rows")

stg_stock_prices: 1228 rows
stg_macro_indicators: 45 rows
dim_ticker: 4 rows
dim_date: 314 rows
fact_stock_price: 1228 rows
fact_macro_indicator: 45 rows
